# 07 · Advanced Type Hints

**Module 10 – Advanced Python: Internals & Metaprogramming** · Estimated time: 75 min

> Builds on Module 05 · 06 (type-hint basics). Here: everything you need to type real libraries and frameworks.

### 🎯 What you will learn
1. Generics in depth: bounds, constraints, **defaults** (PEP 696), variance
2. Typing decorators: **`ParamSpec`**, `Concatenate`; variadic generics: **`TypeVarTuple`**
3. `@overload` for functions whose return type depends on arguments
4. `Literal`, `LiteralString`, `Final`, `ClassVar`, `@final`, `@override`
5. `TypedDict` in depth: `Required`/`NotRequired`, `ReadOnly` (3.13), `closed`/`extra_items` (3.15), `**kwargs: Unpack[...]`
6. Narrowing: `isinstance`, **`TypeIs`** (3.13) / `TypeGuard`, **exhaustiveness** with `Never` + `assert_never`
7. `Annotated` (metadata used by Pydantic/FastAPI/Typer), `NewType`
8. Protocols (generic & runtime-checkable), `Self`, `dataclass_transform`
9. `TYPE_CHECKING`, `cast`, `reveal_type`, runtime introspection (`get_type_hints`, `get_origin`, `get_args`)
10. Configuring **mypy** strict mode; pyright and ty

> We run **mypy** on small files to show the errors a type checker catches **without running the code**.

In [1]:
import subprocess, sys, textwrap, pathlib

def mypy(code: str) -> None:
    """Type-check a snippet with mypy and print its findings."""
    path = pathlib.Path("snippet.py")
    path.write_text(textwrap.dedent(code))
    out = subprocess.run([sys.executable, "-m", "mypy", str(path), "--strict", "--no-color-output",
                          "--no-error-summary", "--python-version", "3.14", "--hide-error-context"],
                         capture_output=True, text=True).stdout
    print(out.strip() or "✅ no type errors")

mypy("def double(x: int) -> int:\n    return x * 2\n\ndouble('3')\n")

snippet.py:4: error: Argument 1 to "double" has incompatible type "str"; expected "int"  [arg-type]


## 1. Generics in depth

### Bounds and constraints

In [2]:
from collections.abc import Sequence, Hashable
from typing import Protocol

class SupportsLessThan(Protocol):
    def __lt__(self, other, /) -> bool: ...

def biggest[T: SupportsLessThan](items: Sequence[T]) -> T:      # BOUND: T must support <
    return max(items)

def concat[S: (str, bytes)](a: S, b: S) -> S:                  # CONSTRAINED: exactly str or bytes
    return a + b

print(biggest([3, 9, 2]), biggest(["pear", "apple"]), concat("ab", "cd"), concat(b"x", b"y"))

9 pear abcd b'xy'


In [3]:
mypy('''
def concat[S: (str, bytes)](a: S, b: S) -> S:
    return a + b
concat("a", b"b")          # mixing str and bytes
''')

snippet.py:4: error: Value of type variable "S" of "concat" cannot be "Sequence[object]"  [type-var]


### 🆕 Type-parameter defaults (PEP 696, Python 3.13)

In [4]:
class Response[T = dict]:               # if not specified, T defaults to dict
    def __init__(self, data: T):
        self.data = data

print(Response.__type_params__[0].__default__)

<class 'dict'>


### Variance: why `list[int]` is not a `list[float]`

| Variance | Meaning | Examples |
|---|---|---|
| **invariant** | `Box[Sub]` is unrelated to `Box[Base]` | mutable containers: `list`, `dict`, `set` |
| **covariant** | `Box[Sub]` is a `Box[Base]` | read-only: `Sequence`, `Mapping` (values), `tuple`, `frozenset`, return types |
| **contravariant** | `Box[Base]` is a `Box[Sub]` | consumers: `Callable` arguments |

In [5]:
mypy('''
from collections.abc import Sequence

def total(xs: list[float]) -> float:
    xs.append(0.5)                 # legal for list[float]...
    return sum(xs)

def total_ro(xs: Sequence[float]) -> float:
    return sum(xs)

ints: list[int] = [1, 2]
total(ints)                        # ❌ would put a float into a list[int]
total_ro(ints)                     # ✅ Sequence is read-only → covariant
''')

snippet.py:12: error: Argument 1 to "total" has incompatible type "list[int]"; expected "list[float]"  [arg-type]
snippet.py:12: note: "list" is invariant -- see https://mypy.readthedocs.io/en/stable/common_issues.html#variance
snippet.py:12: note: Consider using "Sequence" instead, which is covariant


💡 **Rule:** accept the most general *read-only* type (`Sequence`, `Iterable`, `Mapping`) in parameters; return precise types.

With PEP 695 syntax, variance is **inferred** automatically from how the type parameter is used.

## 2. Typing decorators: `ParamSpec` and `Concatenate`

A decorator typed as `Callable[..., Any]` loses the wrapped function's signature. `ParamSpec` (`**P`) preserves it:

In [6]:
import functools, time
from collections.abc import Callable

def timed[**P, R](func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"{func.__name__} took {time.perf_counter() - start:.4f}s")
    return wrapper

@timed
def greet(name: str, excited: bool = False) -> str:
    return f"Hello {name}{'!' if excited else '.'}"

print(greet("Asha", excited=True))

greet took 0.0000s
Hello Asha!


In [7]:
mypy('''
import functools
from collections.abc import Callable

def timed[**P, R](func: Callable[P, R]) -> Callable[P, R]:
    @functools.wraps(func)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        return func(*args, **kwargs)
    return wrapper

@timed
def greet(name: str, excited: bool = False) -> str:
    return name

greet("Asha", excited="yes")     # the checker still knows the real signature
''')

snippet.py:15: error: Argument "excited" to "greet" has incompatible type "str"; expected "bool"  [arg-type]


`Concatenate` types decorators that **add or remove** a parameter, e.g. injecting a database connection:

In [8]:
from typing import Concatenate

class DB:
    def query(self, sql: str) -> list[str]:
        return [f"row for {sql!r}"]

def with_db[**P, R](func: Callable[Concatenate[DB, P], R]) -> Callable[P, R]:
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
        return func(DB(), *args, **kwargs)          # inject the first argument
    return wrapper

@with_db
def find_user(db: DB, user_id: int) -> list[str]:
    return db.query(f"SELECT * FROM users WHERE id = {user_id}")

print(find_user(7))                                  # callers don't pass db

["row for 'SELECT * FROM users WHERE id = 7'"]


### Variadic generics: `TypeVarTuple` (`*Ts`)

In [9]:
def call_with[*Ts, R](func: Callable[[*Ts], R], *args: *Ts) -> R:
    return func(*args)

def describe(name: str, age: int) -> str:
    return f"{name} is {age}"

print(call_with(describe, "Asha", 30))

Asha is 30


## 3. `@overload`: return types that depend on arguments

In [10]:
from typing import overload, Literal

@overload
def fetch(raw: Literal[True]) -> bytes: ...
@overload
def fetch(raw: Literal[False] = ...) -> str: ...
def fetch(raw: bool = False) -> str | bytes:          # the single real implementation
    data = "payload"
    return data.encode() if raw else data

print(type(fetch()), type(fetch(raw=True)))

<class 'str'> <class 'bytes'>


In [11]:
mypy('''
from typing import overload, Literal
@overload
def fetch(raw: Literal[True]) -> bytes: ...
@overload
def fetch(raw: Literal[False] = ...) -> str: ...
def fetch(raw: bool = False) -> str | bytes:
    return b"x" if raw else "x"

fetch(raw=True).upper()          # bytes.upper exists, fine
fetch().decode()                 # ❌ str has no .decode
''')

snippet.py:11: error: "str" has no attribute "decode"; maybe "encode"?  [attr-defined]


## 4. `Literal`, `LiteralString`, `Final`, `ClassVar`, `@final`, `@override`

In [12]:
mypy('''
from typing import Literal, LiteralString, Final, ClassVar, final, override

Mode = Literal["r", "w", "a"]
def open_file(path: str, mode: Mode) -> None: ...
open_file("x.txt", "rw")                         # ❌ not one of the literals

def run_sql(query: LiteralString) -> None: ...  # only string LITERALS: protects against SQL injection
user_input = input()
run_sql("SELECT * FROM users")                  # ✅
run_sql(f"SELECT * FROM users WHERE name = '{user_input}'")   # ❌ built from untrusted data

MAX_SIZE: Final = 100
MAX_SIZE = 200                                   # ❌ Final can't be reassigned

class Counter:
    instances: ClassVar[int] = 0                 # a class-level attribute, not per instance

@final
class Settings: ...
class MySettings(Settings): ...                  # ❌ can't subclass a @final class

class Base:
    def save(self) -> None: ...
class Child(Base):
    @override
    def saev(self) -> None: ...                  # ❌ typo: overrides nothing
''')

snippet.py:6: error: Argument 2 to "open_file" has incompatible type "Literal['rw']"; expected "Literal['r', 'w', 'a']"  [arg-type]
snippet.py:14: error: Cannot assign to final name "MAX_SIZE"  [misc]
snippet.py:21: error: Cannot inherit from final class "Settings"  [misc]
snippet.py:27: error: Method "saev" is marked as an override, but no base method was found with this name  [misc]


> ⚠️ Notice mypy did **not** flag the `run_sql(f"...")` line: mypy currently treats `LiteralString` like plain `str`. **pyright** (and Pylance) enforce it fully. Support for newer typing features varies between checkers, so check your checker's docs.

## 5. `TypedDict` in depth

In [13]:
from typing import TypedDict, Required, NotRequired, ReadOnly, Unpack

class Movie(TypedDict, total=False):        # total=False: keys optional by default...
    title: Required[str]                    # ...except this one
    year: int
    rating: NotRequired[float]

class User(TypedDict):
    id: ReadOnly[int]                       # 🆕 3.13: can't be modified through this type
    name: str

class Options(TypedDict, total=False):
    timeout: float
    retries: int

def request(url: str, **kwargs: Unpack[Options]) -> str:        # PEP 692: precise **kwargs
    return f"GET {url} with {kwargs}"

print(request("https://x.io", timeout=2.5, retries=3))

GET https://x.io with {'timeout': 2.5, 'retries': 3}


In [14]:
mypy('''
from typing import TypedDict, ReadOnly, NotRequired, Unpack

class User(TypedDict):
    id: ReadOnly[int]
    name: str
    email: NotRequired[str]

u: User = {"id": 1, "name": "Asha"}
u["id"] = 2                              # ❌ read-only key
u["nmae"] = "x"                          # ❌ typo in key

class Options(TypedDict, total=False):
    timeout: float
def request(url: str, **kwargs: Unpack[Options]) -> None: ...
request("x", timout=3)                   # ❌ unexpected keyword
''')

snippet.py:10: error: ReadOnly TypedDict key "id" TypedDict is mutated  [typeddict-readonly-mutated]
snippet.py:11: error: TypedDict "User" has no key "nmae"  [typeddict-unknown-key]
snippet.py:16: error: Unexpected keyword argument "timout" for "request"; did you mean "timeout"?  [call-arg]


### 🆕 Closed TypedDicts and `extra_items` (PEP 728, Python 3.15)

By default a TypedDict may contain **extra** keys at runtime. 3.15 lets you forbid them or type them:

In [15]:
class Point(TypedDict, closed=True):         # no other keys allowed
    x: int
    y: int

class Labels(TypedDict, extra_items=str):    # any extra keys must have str values
    name: str

print(Point.__closed__, Labels.__extra_items__)

True <class 'str'>


## 6. Narrowing and exhaustiveness

Type checkers **narrow** a union after checks like `isinstance`, `is None`, `in`, `match`. Custom checks use **`TypeIs`**:

In [16]:
from typing import TypeIs, Never, assert_never

def is_str_list(val: list[object]) -> TypeIs[list[str]]:       # 🆕 3.13 (TypeGuard: older, one-directional)
    return all(isinstance(x, str) for x in val)

def shout(items: list[object]) -> str:
    if is_str_list(items):
        return " ".join(items).upper()       # items is list[str] here
    return "not all strings"

print(shout(["a", "b"]), "|", shout(["a", 1]))

A B | not all strings


### Exhaustiveness checking with `assert_never`

In [17]:
from enum import Enum

class Shape(Enum):
    CIRCLE = "circle"
    SQUARE = "square"
    TRIANGLE = "triangle"

def area_formula(shape: Shape) -> str:
    match shape:
        case Shape.CIRCLE:
            return "πr²"
        case Shape.SQUARE:
            return "a²"
        case Shape.TRIANGLE:
            return "½bh"
        case _:
            assert_never(shape)              # unreachable if every case is handled

print(area_formula(Shape.SQUARE))

a²


In [18]:
mypy('''
from enum import Enum
from typing import assert_never

class Shape(Enum):
    CIRCLE = "circle"
    SQUARE = "square"
    TRIANGLE = "triangle"       # newly added member...

def area_formula(shape: Shape) -> str:
    match shape:
        case Shape.CIRCLE:
            return "πr²"
        case Shape.SQUARE:
            return "a²"
        case _:
            assert_never(shape)  # ❌ mypy reports the forgotten TRIANGLE case
''')

snippet.py:17: error: Argument 1 to "assert_never" has incompatible type "Literal[Shape.TRIANGLE]"; expected "Never"  [arg-type]


## 7. `Annotated` and `NewType`

`Annotated[T, metadata...]` attaches **extra information** to a type. Type checkers see just `T`; libraries read the metadata at runtime. This is the foundation of **Pydantic**, **FastAPI**, **Typer** and SQLAlchemy:

In [19]:
from typing import Annotated, get_type_hints, NewType

class MaxLen:
    def __init__(self, n: int):
        self.n = n

def validate(func):
    hints = get_type_hints(func, include_extras=True)
    def wrapper(**kwargs):
        for name, value in kwargs.items():
            for meta in getattr(hints.get(name), "__metadata__", ()):
                if isinstance(meta, MaxLen) and len(value) > meta.n:
                    raise ValueError(f"{name} longer than {meta.n}")
        return func(**kwargs)
    return wrapper

@validate
def register(username: Annotated[str, MaxLen(8)], bio: str = "") -> str:
    return f"registered {username}"

print(register(username="asha"))
try:
    register(username="averyveryverylongname")
except ValueError as e:
    print("ValueError:", e)

registered asha
ValueError: username longer than 8


In [20]:
UserId = NewType("UserId", int)          # a distinct type for the checker, a plain int at runtime
OrderId = NewType("OrderId", int)

def get_user(uid: UserId) -> str:
    return f"user {uid}"

print(get_user(UserId(5)), type(UserId(5)))
mypy('''
from typing import NewType
UserId = NewType("UserId", int)
OrderId = NewType("OrderId", int)
def get_user(uid: UserId) -> str: return str(uid)
get_user(OrderId(5))                  # ❌ mixing up IDs is caught
''')

user 5 <class 'int'>


snippet.py:6: error: Argument 1 to "get_user" has incompatible type "OrderId"; expected "UserId"  [arg-type]


## 8. Protocols, `Self`, `dataclass_transform`

In [21]:
from typing import Protocol, runtime_checkable, Self, dataclass_transform

class Comparable(Protocol):
    def __lt__(self, other: Self, /) -> bool: ...

class Repository[T](Protocol):                  # a GENERIC protocol
    def get(self, id: int) -> T | None: ...
    def add(self, item: T) -> None: ...

class InMemoryRepo[T]:
    def __init__(self) -> None:
        self._items: dict[int, T] = {}
    def get(self, id: int) -> T | None:
        return self._items.get(id)
    def add(self, item: T) -> None:
        self._items[len(self._items)] = item

def first_item[T](repo: Repository[T]) -> T | None:   # accepts ANY class with get/add (no inheritance)
    return repo.get(0)

repo = InMemoryRepo[str]()
repo.add("hello")
print(first_item(repo))

class QueryBuilder:
    def __init__(self) -> None:
        self.parts: list[str] = []
    def where(self, cond: str) -> Self:          # returns the SUBCLASS type when subclassed
        self.parts.append(cond)
        return self

print(QueryBuilder().where("a > 1").where("b < 2").parts)

hello
['a > 1', 'b < 2']


`@dataclass_transform` tells type checkers that *your* decorator/base class/metaclass creates dataclass-like `__init__`s, which is how Pydantic, SQLAlchemy and attrs get correct type checking:

In [22]:
@dataclass_transform()
def model(cls):
    import dataclasses
    return dataclasses.dataclass(cls)

@model
class Product:
    name: str
    price: float

print(Product("pen", 2.5))

Product(name='pen', price=2.5)


## 9. `TYPE_CHECKING`, `cast`, `reveal_type`, and runtime introspection

In [23]:
from typing import TYPE_CHECKING, cast, get_origin, get_args, Any

if TYPE_CHECKING:                         # only evaluated by type checkers, never at runtime
    from decimal import Decimal           # avoid import cycles / heavy imports

def total(prices: list["Decimal"]) -> "Decimal": ...

raw: Any = {"port": "8080"}
port = cast(dict[str, str], raw)["port"]  # tell the checker "trust me" (no runtime effect!)

hint = dict[str, list[int | None]]
print(get_origin(hint), get_args(hint), get_args(get_args(hint)[1]))
def signup(username: Annotated[str, MaxLen(8)], age: int | None = None) -> None: ...
print(get_type_hints(signup, include_extras=True))

<class 'dict'> (<class 'str'>, list[int | None]) (int | None,)
{'username': typing.Annotated[str, <__main__.MaxLen object at 0x109af9310>], 'age': int | None, 'return': <class 'NoneType'>}


`reveal_type(expr)` makes the type checker print the inferred type (at runtime it prints too, since 3.11).

## 10. Configuring type checkers

```toml
# pyproject.toml
[tool.mypy]
python_version = "3.14"
strict = true                     # enables all the optional strictness flags
warn_unreachable = true
plugins = ["pydantic.mypy"]       # plugins teach mypy about libraries
[[tool.mypy.overrides]]
module = ["legacy.*"]
ignore_errors = true
```

| Checker | Notes |
|---|---|
| **mypy** | the original, plugins, very configurable |
| **pyright** / Pylance | fast, powers VS Code's IntelliSense |
| **ty** (Astral) | very fast (Rust); `uv check` runs it (Module 08 · 04) |
| **pyrefly** (Meta) | fast (Rust) |

Silencing: `# type: ignore[error-code]` for a single line; prefer fixing or narrowing.

## 📝 Summary
- Generics: bounds `T: Proto`, constraints `T: (A, B)`, defaults `T = X`, inferred variance.
- `ParamSpec`/`Concatenate` type decorators; `TypeVarTuple` types variadic functions; `@overload` for argument-dependent returns.
- `Literal`, `LiteralString`, `Final`, `ClassVar`, `@final`, `@override` make intent checkable.
- `TypedDict` with `Required`/`NotRequired`/`ReadOnly`/`closed`/`extra_items` and `Unpack` for kwargs.
- `TypeIs` + `assert_never` give safe narrowing and exhaustiveness; `Annotated` carries metadata for frameworks.

## 🏋️ Practice
1. Type a `retry(times: int)` decorator factory with `ParamSpec` so the decorated function keeps its signature.
2. Write overloads for `parse(value: str, as_int: Literal[True]) -> int` / `Literal[False] -> str`.
3. Define a generic `Stack[T]` protocol and two implementations; write a function accepting any `Stack[int]`.
4. Add a new member to an `Enum` and let `assert_never` + mypy tell you which `match` statements need updating.

<details><summary>Show solution for 1</summary>

```python
import functools
from collections.abc import Callable

def retry[**P, R](times: int) -> Callable[[Callable[P, R]], Callable[P, R]]:
    def decorator(func: Callable[P, R]) -> Callable[P, R]:
        @functools.wraps(func)
        def wrapper(*args: P.args, **kwargs: P.kwargs) -> R:
            for attempt in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception:
                    if attempt == times - 1:
                        raise
            raise RuntimeError("unreachable")
        return wrapper
    return decorator

@retry(3)
def fetch(url: str) -> str:
    return url
print(fetch("https://x.io"))
```
</details>